# Laboratorium 2 — funkcje kwadratowe i układy liniowe

**Metody Optymalizacji · 2026**

Uzupełnij `...`, wykonując komórki po kolei. Przy pytaniach zapisz krótkie wnioski.
Porównamy dokładny GD z teorią z ćwiczeń oraz z metodami LU i Cholesky’ego.
Potrzebne biblioteki: NumPy, Matplotlib, SciPy i CVXOPT.

In [ ]:
%pip install cvxopt

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import linalg, sparse
from time import perf_counter

from scipy.sparse.linalg import splu          # metoda LU do rzadkich macierzy
from cvxopt import matrix, spmatrix, cholmod  # metoda Chol do rzadkich macierzy

## 1. Dokładny krok GD

Na początku przypomnij zadanie 3.1.
Co mierzy norma $\|\cdot\|_A$? Od czego zależy poprawa w pojedynczym kroku?

Dla $f(x)=\frac12x^\top Ax-b^\top x$, gdzie $A=A^\top>0$, mamy
$$\nabla f(x)=Ax-b,\qquad Ax^\star=b.$$
Zapiszemy GD za pomocą residuum $r_k=b-Ax_k$:
$$\alpha_k=\frac{r_k^\top r_k}{r_k^\top Ar_k},\qquad x_{k+1}=x_k+\alpha_k r_k.$$
Uzupełnij pięć miejsc w poniższej pętli. We wszystkich przykładach $b\neq0$.
Kończymy, gdy $\|r_k\|_2\leq\varepsilon$, lub po wyczerpaniu limitu.

Gotowa część zapisuje punkty `X`, normy residuum `R` i czasy `T` w sekundach.
Wynik zawiera też ostatni punkt `x` i `status`. Parametry `keep_history` i `time_limit`
wykorzystamy w dużym przykładzie; na razie zostaw ich wartości domyślne.

In [ ]:
def gd(A, b, x0, eps=1e-6, max_iter=20_000,
       keep_history=True, time_limit=np.inf):
    start = perf_counter()
    x = np.array(x0, dtype=float)
    Ax = A @ x0
    r = b - Ax  # residuum w punkcie x
    X = [x.copy()] if keep_history else []
    R = [np.linalg.norm(r)]
    T = [0.]
    status = "limit iteracji"
    for k in range(max_iter):
        if not np.isfinite(R[-1]):
            status = "nieskończone residuum"
            break
        if R[-1] <= eps:
            break
        if perf_counter() - start >= time_limit:
            status = "limit czasu"
            break
        Ar = A @ r
        alpha = (r @ r) / (r @ Ar)  # dokładny krok
        x = x + alpha * r  # nowy punkt
        Ax = Ax + alpha * Ar # aktualizacja Ax
        r = b - Ax  # residuum w nowym punkcie
        if keep_history:
            X.append(x.copy())
        R.append(np.linalg.norm(r))
        T.append(perf_counter() - start)
    if R[-1] <= eps:
        status = "OK"
    return {"x": x, "X": np.array(X), "R": np.array(R),
            "T": np.array(T), "status": status}

In [ ]:
A = np.eye(2)
b = np.array([1., 2.])
result = gd(A, b, np.zeros(2))
assert np.allclose(result["x"], b)
assert len(result["R"]) - 1 == 1
assert result["status"] == "OK"
print("Liczba kroków:", len(result["R"]) - 1)

Dlaczego dla $A=I$ wystarcza jeden krok? Co się stanie, jeśli zaczniemy w samym minimum?

## 2. Geometria i oszacowanie zbieżności

Z zadania 3.1:
$$E_k:=\|x_k-x^\star\|_A^2=2\bigl(f(x_k)-f(x^\star)\bigr),\qquad
\frac{E_k}{E_0}\leq q^{2k},\quad q=\frac{\kappa(A)-1}{\kappa(A)+1}.$$
Uzupełnij funkcję obliczającą **kwadrat** normy $\|\cdot\|_A$ błędu (nazywaną też normą energetyczną).

In [ ]:
def energy(A, x, x_star):
    e = x - x_star
    return e @ (A @ e)


assert np.isclose(energy(np.diag([1., 10.]), np.ones(2), np.zeros(2)), 11.)

Sprawdźmy zachowanie dla konkretnego przykładu: $A=\operatorname{diag}(1,\kappa)$ i $x^\star=(1,1)$.
Punkty początkowe wybierzemy jako $x_0=x^\star+v/\|v\|_2$, gdzie
$$v\in\{(1,0),\ (1,1),\ (1,1/\kappa)\}.$$
Czy duże $\kappa$ musi oznaczać wiele iteracji dla każdego z tych startów?

Poniższy kod rysujący jest gotowy. Poziomice pokazują $f-f(x^\star)$, a trajektorie
pierwszych 50 kroków. Drugi wykres pokazuje pełne przebiegi $E_k/E_0$ i oszacowanie z teorii.
Zera są pomijane na osi logarytmicznej; dokładność sprawdź również w wypisanych wynikach.

In [ ]:
def positive(values):
    values = np.asarray(values)
    return np.where(values > 0, values, np.nan)

def add_slope(ax, y, color, label_pos=0.5):
    """Fits log10(y) ~ a*k + b over finite points and writes the slope near the line."""
    k = np.arange(len(y))
    ok = np.isfinite(y) & (y > 1e-14)
    if ok.sum() < 2:
        return None
    a, b = np.polyfit(k[ok], np.log10(y[ok]), 1)
    kk = k[ok]
    j = kk[int(label_pos * (len(kk) - 1))]
    ax.annotate(f"nachylenie = {a:.3g}\n(×{10**a:.3g}/krok)", (j, 10**(a*j + b)),
                xytext=(8, 8), textcoords="offset points", fontsize=7, color=color)
    return a

def geometry(kappa, directions, direction_labels, show_slopes=False):
    A = np.diag([1., kappa])
    x_star = np.ones(2)
    b = A @ x_star
    fig = plt.figure(figsize=(15, 4))
    outer = fig.add_gridspec(1, 3, width_ratios=[1, 1, 1])
    inner = outer[1].subgridspec(2, 1, hspace=0.15)
    ax_traj = fig.add_subplot(outer[0])
    ax_x0 = fig.add_subplot(inner[0])
    ax_x1 = fig.add_subplot(inner[1], sharex=ax_x0)
    ax_err = fig.add_subplot(outer[2])
    axes = [ax_traj, ax_err]
    grid = np.meshgrid(np.linspace(-0.2, 2.2, 200), np.linspace(-0.2, 2.2, 200))
    u = grid[0]
    v = grid[1]
    axes[0].contour(u, v, ((u-1)**2 + kappa*(v-1)**2)/2,
                    levels=np.geomspace(0.001, kappa, 12), colors="grey", alpha=0.4)
    longest = 0
    for i in range(len(directions)):
        x0 = x_star + directions[i] / np.linalg.norm(directions[i])
        result = gd(A, b, x0)
        X = result["X"]
        E = np.array([energy(A, x, x_star) for x in X])
        axes[0].plot(X[:51, 0], X[:51, 1], ".-", label=direction_labels[i], markersize=3)
        ax_x0.plot(X[:, 0], ".-", label=direction_labels[i], markersize=2)
        ax_x1.plot(X[:, 1], ".-", label=direction_labels[i], markersize=2)
        line, = axes[1].semilogy(positive(E / E[0]), ".-", label=direction_labels[i], markersize=2)
        if show_slopes:
            add_slope(axes[1], positive(E / E[0]), line.get_color(), 0.3 + 0.2*i)
        longest = max(longest, len(E))
        print(f"κ = {kappa:3g} | {direction_labels[i]:16s} | kroki: {len(E)-1:4d} | "
              f"residuum: {result['R'][-1]:.2e} | {result['status']}")
    q = (kappa - 1) / (kappa + 1)
    bound = positive(q**(2*np.arange(longest)))
    axes[1].semilogy(bound, "k--", label="oszacowanie q^(2k)")
    if show_slopes:
        add_slope(axes[1], bound, "k", 0.8)
    axes[0].plot(1, 1, "k*", markersize=10)
    axes[0].set(xlabel="x₀", ylabel="x₁", title=f"κ = {kappa}, pierwsze 50 kroków")
    axes[0].set_aspect("equal")
    axes[1].set(xlabel="Numer iteracji k", ylabel="Eₖ / E₀", ylim=(1e-14, 2),
               title="Relatywna energia w kolejnych iteracjach")
    ax_x0.axhline(1, color="k", ls="--", lw=0.8)
    ax_x1.axhline(1, color="k", ls="--", lw=0.8)
    ax_x0.set(ylabel="x₀", title="Zmienne w kolejnych iteracjach")
    ax_x1.set(xlabel="Numer iteracji k", ylabel="x₁")
    plt.setp(ax_x0.get_xticklabels(), visible=False)
    for ax in axes + [ax_x0, ax_x1]:
        ax.legend(fontsize=8)
        ax.grid(alpha=0.2)
    fig.tight_layout()
    plt.show()

In [ ]:
for kappa in [1., 10., 100]:
    geometry(
        kappa,
        directions = [np.array([1., 0.]), np.array([1., 1.]), np.array([1., 1/kappa])],
        direction_labels = ["v = (1, 0)", "v = (1, 1)", "v = (1, 1/κ)"]
    )

In [ ]:
kappa = 40
q = (kappa - 1) / (kappa + 1)
print(f"κ = {kappa:g} | q² = {q**2:.4f} | nachylenie = log10(q²) = {np.log10(q**2):.4f}")

geometry(
    kappa,
    directions = [np.array([1., 1/kappa]), np.array([1., 2/kappa]), np.array([1., 4/kappa])],
    direction_labels = ["v = (1, 1/κ)", "v = (1, 2/κ)", "v = (1, 4/κ)"],
    show_slopes=True
)

Który start daje jeden krok, a który przebieg osiąga równość w oszacowaniu? Wyjaśnij oba przypadki.
Czy szybszy spadek błędu przeczy oszacowaniu z ćwiczeń? Co zmienia zwiększenie $\kappa$?

## 3. Dokładny krok a warunki Wolfe’a

**Zadanie obowiązkowe — rozwiązujemy na sali.** Niech $p$ będzie dowolnym kierunkiem spadku
dla naszej funkcji kwadratowej, a $\phi(\alpha)=f(x+\alpha p)$.
Pokaż, że dokładny krok $\alpha^\star=\arg\min_{\alpha\geq0}\phi(\alpha)$ jest dodatni
i spełnia oba warunki Wolfe’a dla $0<c_1\leq\frac12$, $c_1<c_2<1$:
$$\phi(\alpha^\star)\leq\phi(0)+c_1\alpha^\star\phi'(0),\qquad
\phi'(\alpha^\star)\geq c_2\phi'(0).$$
Sprawdź także silny warunek krzywizny:
$$|\phi'(\alpha^\star)|\leq c_2|\phi'(0)|.$$

Na Lab 1 minimum $\phi(\alpha)=(1-\alpha)^2+(1-\alpha)^{20}$ nie spełniało Armijo
dla $c_1=0{,}1$. Dlaczego nie ma sprzeczności? Czy sam backtracking Armijo gwarantuje krzywiznę?

## 4. GD, LU i Cholesky — czas oraz dokładność

Porównamy GD z dwiema metodami bezpośrednimi. Ich wyniki także podlegają błędom zaokrągleń.
Najpierw uruchom gotowy generator $A=Q\Lambda Q^\top$ oraz funkcje rozkładu i rozwiązania.
`factor` wykonuje rozkład raz; `solve_factor` pozwala używać go dla kolejnych prawych stron.

In [ ]:
def make_spd(values, seed=2026):
    rng = np.random.default_rng(seed)
    n = len(values)
    qr = np.linalg.qr(rng.normal(size=(n, n)))
    Q = qr[0]
    A = (Q * values) @ Q.T
    return (A + A.T) / 2


def factor(A, method):
    if method == "LU":
        return linalg.lu_factor(A)
    return linalg.cho_factor(A)


def solve_factor(F, b, method):
    if method == "LU":
        return linalg.lu_solve(F, b)
    return linalg.cho_solve(F, b)

In [ ]:
def direct(A, b, method, repeats=3):
    times = []
    for repetition in range(repeats):
        start = perf_counter()
        F = factor(A, method)
        x = solve_factor(F, b, method)
        times.append(perf_counter() - start)
    return {"x": x, "time": np.median(times)}

Użyj gotowej funkcji `benchmark()`. Metody działają kolejno, po trzy razy.
Dla GD wybieramy przebieg o środkowym czasie zakończenia, a dla rozkładów medianę czasów.
Pomiar LU/Cholesky’ego obejmuje rozkład i rozwiązanie; generowanie danych i wykresy są poza pomiarem.
W czasie GD mieści się także sprawdzanie stopu i zapis historii.

Na wykresach pokazujemy normę residuum i względny błąd rozwiązania. Linie pionowe
oznaczają zakończenie metod bezpośrednich. Krzywe GD są schodkowe: między iteracjami
dostępne jest ostatnie obliczone przybliżenie.
Po zakończeniu GD dostępny pozostaje jego ostatni wynik.

In [ ]:
def print_header():
    print(f"{'metoda':<16} | {'czas [s]':>10} | {'norma residuum':>15} | "
          f"{'wzgl. błąd':>12} | {'kroki':>6} | status")

def print_row(method, elapsed, residual, error=None, steps=None, status="—"):
    error_text = "—" if error is None else f"{error:.2e}"
    steps_text = "—" if steps is None else str(steps)
    print(f"{method:<16} | {elapsed:10.3e} | {residual:15.2e} | "
          f"{error_text:>12} | {steps_text:>6} | {status}")
    
def benchmark(n, kappa, eps=1e-8):
    A = make_spd(np.linspace(1., kappa, n))
    x_star = np.ones(n)
    b = A @ x_star
    start = np.zeros(n)
    plot = plt.subplots(1, 2, figsize=(11, 4))
    axes = plot[1]
    print(f"n = {n}, κ = {kappa}; tolerancja bezwzględna GD = {eps:g}")
    print_header()
    runs = [gd(A, b, start, eps = eps) for repetition in range(3)]
    runs.sort(key=lambda run: run["T"][-1])
    result = runs[1]
    error = np.linalg.norm(result["X"] - x_star, axis=1) / np.linalg.norm(x_star)
    label = "GD"
    axes[0].step(result["T"], positive(result["R"]), where="post", label=label)
    axes[1].step(result["T"], positive(error), where="post", label=label)
    print_row(label, result["T"][-1], result["R"][-1], error[-1],
              steps=len(error)-1, status=result["status"])
    methods = ["LU", "Cholesky"]
    for i in range(len(methods)):
        method = methods[i]
        result = direct(A, b, method)
        residual = np.linalg.norm(b - A @ result["x"])
        error = np.linalg.norm(result["x"] - x_star) / np.linalg.norm(x_star)
        measures = [residual, error]
        for j in range(2):
            axes[j].plot(result["time"], positive(measures[j]), "o", color=f"C{i+3}", label=method)
            axes[j].axvline(result["time"], color=f"C{i+3}", linestyle=":")
        print_row(method, result["time"], residual, error)
    axes[0].set_ylabel("Norma residuum")
    axes[1].set_ylabel("Względny błąd rozwiązania")
    for ax in axes:
        ax.set(xlabel="Czas [s]", yscale="log", title=f"n = {n}, κ = {kappa}")
        ax.ticklabel_format(axis="x", style="sci", scilimits=(-2, 2), useMathText=True)
        ax.locator_params(axis="x", nbins=5)
        ax.legend(fontsize=8)
        ax.grid(alpha=0.2)
    plot[0].tight_layout()
    plt.show()

In [ ]:
benchmark(300, 5., eps=1e-8)

Jak dobre przybliżenie ma GD w chwili zakończenia LU i Cholesky’ego?
Czy mniejsze residuum zawsze oznacza mniejszy błąd rozwiązania?
Czy residuum GD musi maleć w każdej iteracji? Która miara postępu musi maleć?

Zmieńmy osobno wymiar i uwarunkowanie. Przed uruchomieniem przewidź wpływ każdej zmiany.

In [ ]:
benchmark(1000, 5., eps=1e-8)

Duży przykład:

In [ ]:
benchmark(7000, 5., eps=1e-8)

In [ ]:
benchmark(300, 100., eps=1e-8)

Co zmieniło się w kosztach metod i uzyskanej dokładności?
Przy krótkich czasach powtórz porównanie; nie przywiązuj wagi do pojedynczych niewielkich różnic.

## 5. Duży układ — wygładzanie danych przestrzennych

Dane $y$ opisują zaszumiony obraz przestrzenny na sześciennej siatce.
Każda próbka ma najwyżej sześciu sąsiadów. Szukamy wartości bliskich danym,
ograniczając jednocześnie różnice między sąsiadami:

$$
\min_x\left\{
\frac12\|x-y\|_2^2+
\frac{\rho}{2}\sum_{\{i,j\}\in E}w_{ij}(x_i-x_j)^2
\right\}.
$$

Zbiór $E$ zawiera nieuporządkowane pary sąsiadów, a dodatnie wagi $w_{ij}$
określają siłę wygładzania. W przykładzie losujemy je z $[0.5,1.5]$.

W przygotowanym kodzie $W$ jest symetryczną macierzą wag połączeń,
$D=\operatorname{diag}(\sum_j W_{ij})$, a $I$ jest macierzą jednostkową.

Zapisz sumę $\sum_{\{i,j\}\in E}w_{ij}(x_i-x_j)^2$ jako $x^\top L_wx$.
Wyznacz gradient funkcji celu i zapisz warunek optymalności jako $Ax=y$.
Uzasadnij, że dla $\rho>0$ macierz $A$ jest symetryczna dodatnio określona.

Uzupełnij dwie linie wyznaczające $L_w$ i $A$.

Pozostały kod jest gotowy. Najpierw porównamy dwa starty GD z rzadkimi metodami
LU i Cholesky’ego na mniejszych siatkach.
Sprawdzimy także liczbę niezerowych elementów macierzy i jej czynników.
Potem powtórzymy porównanie na większej siatce.

In [ ]:
def volume_problem(m, rho=1.):
    rng = np.random.default_rng(2026)
    n = m**3
    grid = np.arange(n).reshape(m, m, m)
    i = np.concatenate([grid[:-1].ravel(), grid[:, :-1].ravel(),
                        grid[:, :, :-1].ravel()])
    j = np.concatenate([grid[1:].ravel(), grid[:, 1:].ravel(),
                        grid[:, :, 1:].ravel()])
    weights = rng.uniform(0.5, 1.5, size=len(i))
    W = sparse.coo_matrix((weights, (i, j)), shape=(n, n)).tocsr()
    W = W + W.T
    degree = np.asarray(W.sum(axis=1)).ravel()
    D = sparse.diags(degree, format="csr")
    I = sparse.eye(n, format="csr")
    L_w = D - W  # macierz członu wygładzającego
    A = I + rho * L_w    # macierz układu wynikającego z warunku optymalności

    q = np.linspace(-1., 1., m)
    radius2 = q[:, None, None]**2 + q[None, :, None]**2 + q[None, None, :]**2
    signal = (radius2 < 0.55**2).astype(float)
    y = signal.ravel() + rng.normal(scale=0.3, size=n)
    return {"A": A, "y": y, "signal": signal}

In [ ]:
def compare_volume(problem, eps, time_limit=2., repeats=3,
                   count_fill=False, direct_methods=("LU", "Cholesky")):
    A = problem["A"]
    y = problem["y"]
    starts = [np.zeros(len(y)), y]
    results = []

    for x0 in starts:
        runs = [
            gd(A, y, x0, eps=eps, keep_history=False, time_limit=time_limit)
            for repetition in range(repeats)
        ]
        runs.sort(key=lambda run: run["T"][-1])
        result = runs[len(runs)//2]
        result["time"] = result["T"][-1]
        result["residual"] = result["R"][-1]
        result["steps"] = len(result["R"])-1
        results.append(result)

    # Przygotowanie formatów poza pomiarem.
    if "LU" in direct_methods:
        A_csc = A.tocsc()
    lower = sparse.tril(A, format="coo")
    A_chol = spmatrix(lower.data, lower.row, lower.col, size=A.shape)
    cholmod.options["supernodal"] = 2

    for method in direct_methods:
        times = []
        for repetition in range(repeats):
            start = perf_counter()
            if method == "LU":
                F = splu(A_csc, permc_spec="MMD_AT_PLUS_A",
                         options={"SymmetricMode": True})
                x = F.solve(y)
            else:
                x_cv = matrix(y)
                F = cholmod.symbolic(A_chol)
                cholmod.numeric(A_chol, F)
                cholmod.solve(F, x_cv)
            elapsed = perf_counter() - start

            if method == "LU" and repetition == repeats - 1 and count_fill:
                print(f"Niezerowe elementy A: {A.nnz:,}; "
                      f"łącznie L i U: {F.L.nnz + F.U.nnz:,}")
            del F

            times.append(elapsed)

        if method == "Cholesky":
            x = np.asarray(x_cv).ravel().copy()

        results.append({
            "method": method,
            "x": x,
            "time": np.median(times),
            "residual": np.linalg.norm(y - A @ x),
            "steps": None,
            "status": "—"
        })

    names = ["GD: x0 = 0", "GD: x0 = y"]
    names += [result["method"] for result in results[2:]]
    print(f"Tolerancja bezwzględna GD: {eps:g}")
    print_header()
    for i in range(len(results)):
        result = results[i]
        print_row(names[i], result["time"], result["residual"],
                  steps=result["steps"], status=result["status"])

    return results

def plot_volume(problem, results, eps):
    m = problem["signal"].shape[0]
    z = np.linspace(-1., 1., m)[m//2]
    print(f"Pokazujemy środkową warstwę siatki w obszarze [-1, 1]³: z = {z:.3f}. "
          "Obliczenia dotyczą całego obrazu 3D.")
    names = ["GD: x0 = 0", "GD: x0 = y"]
    names += [result["method"] for result in results[2:]]
    colors = ["C0", "C1"]
    colors += ["C2" if name == "LU" else "C3" for name in names[2:]]

    volumes = [problem["signal"], problem["y"].reshape(m, m, m)]
    volumes += [result["x"].reshape(m, m, m) for result in results]
    titles = ["Sygnał bez szumu", "Dane y", "$x_0=0$", "$x_0=y$"]
    titles += [
        "Chol" if result["method"] == "Cholesky" else result["method"]
        for result in results[2:]
    ]

    # Numery pól: pierwszy wiersz 0, 1, 2; drugi wiersz 3, 4, 5.
    positions = [0, 1, 3, 4]
    positions += [
        2 if result["method"] == "Cholesky" else 5
        for result in results[2:]
    ]

    plot = plt.subplots(2, 3, figsize=(10, 6))
    axes = plot[1].ravel()
    for ax in axes:
        ax.set_axis_off()

    for i in range(len(volumes)):
        ax = axes[positions[i]]
        ax.imshow(volumes[i][:, :, m//2],
                  cmap="gray", vmin=0., vmax=1.)
        ax.set_title(titles[i])

    plot[0].tight_layout()
    plt.show()

    plot = plt.subplots(1, 2, figsize=(12, 4))
    axes = plot[1]
    end = max(result["time"] for result in results)
    zoom_end = max(results[0]["time"], results[1]["time"], 1e-6)

    for j in range(2):
        limit = end if j == 0 else zoom_end

        for i in range(2):
            result = results[i]
            axes[j].step(result["T"], positive(result["R"]), where="post",
                         color=colors[i], label=names[i])
            axes[j].hlines(
                positive(result["residual"]), result["time"], limit,
                colors=colors[i], linestyles=":", alpha=0.6
            )

        for i in range(2, len(results)):
            result = results[i]
            if result["time"] <= 1.05*limit:
                axes[j].plot(result["time"], positive(result["residual"]),
                             "o", color=colors[i], label=names[i])
                axes[j].axvline(result["time"],
                                color=colors[i], linestyle=":")

        axes[j].axhline(eps, color="grey", linestyle="--",
                       label=f"ε = {eps:g}")
        axes[j].set(xlabel="Czas [s]", ylabel="Norma residuum",
                    yscale="log", xlim=(-0.02*limit, 1.05*limit))
        axes[j].grid(alpha=0.2)
        axes[j].legend(fontsize=8)

    axes[0].set_title("Pełne porównanie")
    axes[1].set_title("Zbliżenie przebiegów GD")
    plot[0].tight_layout()
    plt.show()

In [ ]:
# Wspólne ustawienia całej sekcji 5.
eps_large = 1e-12
time_limit_large = 2.
repeats_large = 3

for m in [12, 24]:
    print(f"\nSiatka {m}³: {m**3:,} niewiadomych")
    problem = volume_problem(m)
    results = compare_volume(
        problem,
        eps=eps_large,
        time_limit=time_limit_large,
        repeats=repeats_large,
        count_fill=True
    )

Podczas rozkładu mogą powstawać niezerowe elementy w miejscach,
w których macierz wejściowa miała zera. Zjawisko to nazywamy wypełnieniem.
Rzadka macierz nie musi więc mieć równie rzadkich czynników.

Teraz zwiększamy rozmiar siatki. Porównujemy GD z punktów początkowych
$x_0=0$ i $x_0=y$ oraz rozwiązania uzyskane przez LU i Cholesky’ego.

Oba przebiegi GD korzystają ze wspólnej tolerancji bezwzględnej `eps_large`
i budżetu `time_limit_large`. Budżet jest sprawdzany między iteracjami.

Pokażemy środkowe przekroje danych i wszystkich wyników we wspólnej skali
kolorów. Na wykresie czasowym punkty oznaczają zakończenie metod
bezpośrednich, a poziome przedłużenia — ostatnie dostępne wyniki GD.

In [ ]:
m = 32
problem = volume_problem(m)

print(f"\nSiatka {m}³: {m**3:,} niewiadomych")
results = compare_volume(
    problem,
    eps=eps_large,
    time_limit=time_limit_large,
    repeats=repeats_large
)

In [ ]:
plot_volume(problem, results, eps=eps_large)

W tym porównaniu LU jest wyraźnie wolniejsze. Na większej siatce porównamy oba starty GD z Choleskym.

In [ ]:
m_big = 80
problem_big = volume_problem(m_big)

print(f"\nSiatka {m_big}³: {m_big**3:,} niewiadomych")
results_big = compare_volume(
    problem_big,
    eps=eps_large,
    time_limit=time_limit_large,
    repeats=1,
    direct_methods=["Cholesky"]
)

In [ ]:
plot_volume(problem_big, results_big, eps=eps_large)

1. Jak zmieniła się liczba niezerowych elementów podczas rozkładu?
   Co oznacza to dla kosztu przechowywania czynników?
2. Czy GD osiągnęło tolerancję w zadanym budżecie?
   Czy otrzymane wygładzenie jest użyteczne?
3. Czy bardzo małe residuum gwarantuje odzyskanie sygnału bez szumu?
   Rozróżnij dokładność rozwiązania zadania optymalizacji i jakość wygładzenia.

# Zadania

Wykorzystaj uzupełnione funkcje z zajęć. Rozwiązania: `MO_02.ipynb`.
Przy wynikach GD podawaj liczbę iteracji, końcową normę residuum i przyczynę zakończenia.

## 1. Residuum a błąd rozwiązania

Dla symetrycznej dodatnio określonej macierzy $A$, $b\neq0$ i $Ax^\star=b$ wyprowadź nierówność
$$\|x-x^\star\|_2\leq\frac{\|b-Ax\|_2}{\lambda_{\text{min}}(A)}.$$
Następnie zbadaj **jeden krok** dokładnego GD dla
$$A=\operatorname{diag}(10^{-4},1),\qquad x^\star=(1,1),\qquad b=Ax^\star,\qquad x_0=0.$$
Porównaj normę residuum i normę błędu rozwiązania po tym kroku. Czy kryterium stopu z tolerancją
$10^{-3}$ zakończyłoby obliczenia? Jak dokładny byłby wynik i dlaczego nie przeczy to nierówności?

## 2. To samo uwarunkowanie, różne widma

Przyjmij $n=100$. Porównaj dwie macierze $A_i=Q\Lambda_iQ^\top$ o $\kappa=100$:

- wartości własne $A_1$ są równomiernie rozmieszczone od $1$ do $100$;
- widmo $A_2$ wynosi $1,10,\ldots,10,100$.

Użyj wspólnej ortonormalnej bazy $Q$; możesz przyjąć $Q=I$. Dla obu problemów wybierz
$x^\star=Q\mathbf1$, $x_0=0$, a prawe strony wyznacz osobno jako $b_i=A_ix^\star$.
Stosuj tolerancję $10^{-6}$ i limit 20 000 iteracji.

Porównaj na wykresie $\|x_k-x^\star\|_{A_i}^2/\|x_0-x^\star\|_{A_i}^2$
z oszacowaniem z góry $(99/101)^{2k}$. Czy wspólne uwarunkowanie oznacza identyczne przebiegi?
Porównaj początkowy spadek błędu i późniejsze nachylenie krzywych w skali logarytmicznej. Czy skupienie wartości własnych przyspieszyło GD w tym eksperymencie? Dlaczego w porównaniu używamy wspólnego $Q$ i tego samego $x^\star$? Nie zakładaj z góry, który przykład będzie szybszy.
Zachowaj dane i wyniki do późniejszego porównania z gradientami sprzężonymi.

## 3. Jedna macierz, wiele obrazów

Na siatce $64^3$ w obszarze $[-1,1]^3$ wykorzystaj wygładzanie z sekcji 5 z $\rho=1$. Macierz $A=I+\rho L_w$ wygeneruj **raz**, zachowując jej wagi przez cały eksperyment.

Przygotuj 20 obrazów. Dla każdego niezależnie wylosuj jednostajnie środek $c\in[-1,1]^3$ oraz promienie $r_1\in(0.1,0.2)$ i $r_2\in(0.3,0.5)$. Ustaw wartość 1 tam, gdzie $r_1<\|p-c\|_2<r_2$, a poza tym 0. Dodaj niezależny szum $N(0,0.3^2)$. Generator z ustalonym ziarnem utwórz raz przed pętlą.

Dla tych samych danych rozwiązuj kolejno $Ax^{(j)}=y^{(j)}$, porównując:

- **Cholesky’ego:** jeden rozkład, wykorzystywany dla wszystkich obrazów;
- **GD ze startu $x_0=0$;**
- **GD ze startu $x_0=y^{(j)}$.**

W obu GD użyj tolerancji bezwzględnej `eps_large`, limitu 20 000 iteracji, `keep_history=False` i `time_limit=np.inf`.

Narysuj trzy krzywe **łącznego czasu przetworzenia pierwszych $k$ obrazów**, $k=1,\ldots,20$. Do czasu Cholesky’ego dolicz jednorazowy koszt przygotowania rozkładu. Generowanie danych i wykresy pomiń w pomiarze; krótkie pomiary powtarzaj.

Dla każdego startu GD ustal, od ilu obrazów Cholesky staje się szybszy; jeśli nie nastąpi to w badanym zakresie, odnotuj ten fakt. Wyjaśnij różnicę między kosztem przygotowania rozkładu a kosztem rozwiązania układu dla kolejnego obrazu. Dla każdego wariantu podaj największą końcową normę residuum spośród 20 obrazów; dla obu startów GD podaj też liczbę przebiegów, które osiągnęły tolerancję.

Dla dwóch wybranych obrazów pokaż przekroje sygnału bez szumu, danych i wyników w warstwie najbliższej środkowi powłoki. Dla danego obrazu zachowaj wspólną warstwę i skalę kolorów.